In [ ]:
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from sklearn.preprocessing import MinMaxScaler
from tqdm import tqdm
from torch.distributions import Beta

# Load and normalize data

In [ ]:
# Load and normalize data
res_df = pd.read_excel("hourly_data.xlsx") #Moroccan data in hourly intervals
com_df = pd.read_excel("commercial_data.xlsx")
com_df['Commercial_load'] = com_df['Commercial_load'] + 250000 # offset to match Moroccan data ranges

loads_df = pd.DataFrame({
    'Residential_load':res_df['Total Power Consumption'][:192*3].values,
    'Commercial_load':com_df['Commercial_load'][:192*3].values
})


In [ ]:
# Normalize data
min_max_scaler = MinMaxScaler()
grad_scaler = torch.amp.GradScaler('cuda')
res_max = 700000
com_max = 600000
res_load_data = (loads_df['Residential_load'].values).reshape(-1, 1)
res_norm_load = min_max_scaler.fit_transform(res_load_data)

com_load_data = (loads_df['Commercial_load'].values).reshape(-1, 1)
com_norm_load = min_max_scaler.fit_transform(com_load_data)

util_res_load_data = (loads_df['Residential_load'].values/res_max).reshape(-1, 1)
util_res_norm_load = min_max_scaler.fit_transform(util_res_load_data)

util_com_load_data = (loads_df['Commercial_load'].values/com_max).reshape(-1, 1)
util_com_norm_load = min_max_scaler.fit_transform(util_com_load_data)

eps = 1e-8

dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(0)
np.random.seed(0)

# Definition of a PPO Agent

In [ ]:
# --- Beta-policy actor + critic ---
class ActorBeta(nn.Module):
    def __init__(self, state_dim, action_dim, hidden=64):
        super().__init__()
        self.fc1 = nn.Linear(state_dim, hidden)
        self.fc2 = nn.Linear(hidden, hidden)
        # produce positive concentration params (alpha, beta) per action dimension
        self.alpha_head = nn.Linear(hidden, action_dim)
        self.beta_head = nn.Linear(hidden, action_dim)

    def forward(self, state):
        x = F.relu(self.fc1(state))
        x = F.relu(self.fc2(x))
        # softplus to ensure positivity, add 1 to avoid alpha~0
        alpha = F.softplus(self.alpha_head(x)) + 1.0
        beta = F.softplus(self.beta_head(x)) + 1.0
        return alpha, beta

class Critic(nn.Module):
    def __init__(self, state_dim, hidden=64):
        super().__init__()
        self.fc1 = nn.Linear(state_dim, hidden)
        self.fc2 = nn.Linear(hidden, hidden)
        self.v = nn.Linear(hidden, 1)
    def forward(self, state):
        x = F.relu(self.fc1(state))
        x = F.relu(self.fc2(x))
        return self.v(x)

In [ ]:
class PPOBetaAgent:
    def __init__(self, state_dim, action_dim, lr=3e-4, gamma=0.99, lam=0.95, clip_eps=0.2,
                 k_epochs=4, minibatch_size=256, entropy_coef=1e-3, max_grad_norm=0.5):
        self.actor = ActorBeta(state_dim, action_dim).to(dev)
        self.critic = Critic(state_dim).to(dev)
        self.optimizer = optim.Adam(list(self.actor.parameters()) + list(self.critic.parameters()), lr=lr)
        self.gamma = gamma
        self.lam = lam
        self.clip_eps = clip_eps
        self.k_epochs = k_epochs
        self.minibatch_size = minibatch_size
        self.entropy_coef = entropy_coef
        self.max_grad_norm = max_grad_norm

        # logging
        self.last_actor_loss = 0.0
        self.last_critic_loss = 0.0
        self.last_entropy = 0.0

    def get_action_and_logp(self, state_np):
        state = torch.tensor(state_np, dtype=torch.float32, device=dev).unsqueeze(0)
        with torch.no_grad():
            alpha, beta = self.actor(state)
            dist = torch.distributions.Beta(alpha, beta)
            action = dist.rsample()
            logp = dist.log_prob(action).sum(dim=-1)
        return action.squeeze(0).cpu().numpy(), logp.item()

    def evaluate_actions(self, states_tensor, actions_tensor):
        # returns log_probs, entropy, values
        alpha, beta = self.actor(states_tensor)
        dist = torch.distributions.Beta(alpha, beta)
        # clamp actions into (0,1) slightly to avoid log(0)
        actions_clamped = actions_tensor.clamp(eps, 1 - eps)
        logp = dist.log_prob(actions_clamped).sum(dim=-1)
        entropy = dist.entropy().sum(dim=-1)  # per-sample entropy
        values = self.critic(states_tensor).squeeze(-1)
        return logp, entropy, values

    def compute_gae(self, rewards, values, dones, last_value):
        values = np.array(values + [last_value], dtype=np.float32)
        rewards = np.array(rewards, dtype=np.float32)
        dones = np.array(dones, dtype=np.float32)
        gae = 0.0
        advantages = np.zeros_like(rewards, dtype=np.float32)
        for t in reversed(range(len(rewards))):
            nonterminal = 1.0 - dones[t]
            delta = rewards[t] + self.gamma * values[t+1] * nonterminal - values[t]
            gae = delta + self.gamma * self.lam * nonterminal * gae
            advantages[t] = gae
        returns = advantages + values[:-1]
        return advantages, returns

    def update(self, states, actions, old_logps, returns, advantages):
        # convert to tensors.
        states = torch.tensor(states, dtype=torch.float32, device=dev)
        actions = torch.tensor(actions, dtype=torch.float32, device=dev)
        old_logps = torch.tensor(old_logps, dtype=torch.float32, device=dev)
        returns = torch.tensor(returns, dtype=torch.float32, device=dev)
        advantages = torch.tensor(advantages, dtype=torch.float32, device=dev)
        advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)

        N = states.shape[0]
        idxs = np.arange(N)

        actor_loss_epoch = 0.0
        critic_loss_epoch = 0.0
        entropy_epoch = 0.0
        updates = 0

        for epoch in range(self.k_epochs):
            np.random.shuffle(idxs)
            for start in range(0, N, self.minibatch_size):
                end = start + self.minibatch_size
                mb_idx = idxs[start:end]
                mb_states = states[mb_idx]
                mb_actions = actions[mb_idx]
                mb_old_logps = old_logps[mb_idx]
                mb_returns = returns[mb_idx]
                mb_advantages = advantages[mb_idx]

                # evaluate current policy
                logp, entropy, values = self.evaluate_actions(mb_states, mb_actions)
                ratios = torch.exp(logp - mb_old_logps)

                surr1 = ratios * mb_advantages
                surr2 = torch.clamp(ratios, 1.0 - self.clip_eps, 1.0 + self.clip_eps) * mb_advantages
                actor_loss = -torch.min(surr1, surr2).mean() - self.entropy_coef * entropy.mean()

                critic_loss = F.mse_loss(values, mb_returns)

                loss = actor_loss + 0.5 * critic_loss

                self.optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(list(self.actor.parameters()) + list(self.critic.parameters()), self.max_grad_norm)
                self.optimizer.step()

                actor_loss_epoch += actor_loss.item()
                critic_loss_epoch += critic_loss.item()
                entropy_epoch += entropy.mean().item()
                updates += 1

        # record averaged metrics
        if updates > 0:
            self.last_actor_loss = actor_loss_epoch / updates
            self.last_critic_loss = critic_loss_epoch / updates
            self.last_entropy = entropy_epoch / updates
        else:
            self.last_actor_loss = 0.0
            self.last_critic_loss = 0.0
            self.last_entropy = 0.0

# Training the agent

In [ ]:
state_dim = 24
action_dim = 24
agent = PPOBetaAgent(state_dim, action_dim,
                     lr=1e-4, gamma=0.99, lam=0.95, clip_eps=0.2,
                     k_epochs=4, minibatch_size=256, entropy_coef=1e-3, max_grad_norm=0.5)

num_episodes = 15000 
rollout_steps = len(res_norm_load) - 24 

rewards_per_episode = []
actor_losses = []
critic_losses = []
entropy_logs = []
adv_mean_logs = []
adv_std_logs = []

for episode in tqdm(range(num_episodes), desc="Training Progress"):
    states_buf = []
    actions_buf = []
    rewards_buf = []
    logps_buf = []
    values_buf = []
    dones_buf = []

    total_reward = 0.0

    # Collect trajectory (on-policy)
    for i in range(rollout_steps):
        state = res_norm_load[i:i+24].flatten()
        action, logp = agent.get_action_and_logp(state)
        reward = -np.abs(action - res_norm_load[i+1:i+25].flatten()).sum() # reward for peak-shaving and valley-filling
        # reward = -np.abs(action - res_utils_diff[i+1:i+25].flatten()).sum() # reward for inter-distribution networks load balancing
        # reward = -np.abs(action - res_norm_load[i+1:i+25].flatten()).sum() - (np.abs(action - res_utils_diff[i+1:i+25].flatten()).sum()) # reward for both
        next_state = res_norm_load[i+1:i+25].flatten()

        # Critic value for state
        with torch.no_grad():
            value = agent.critic(torch.tensor(state, dtype=torch.float32, device=dev).unsqueeze(0)).item()

        states_buf.append(state)
        actions_buf.append(action)
        rewards_buf.append(reward)
        logps_buf.append(logp)
        values_buf.append(value)
        dones_buf.append(0.0)

        total_reward += reward

    # bootstrap last value
    last_value = agent.critic(torch.tensor(next_state, dtype=torch.float32, device=dev).unsqueeze(0)).item()
    advantages, returns = agent.compute_gae(rewards_buf, values_buf, dones_buf, last_value)

    # logging advantage stats
    adv_mean_logs.append(float(np.mean(advantages)))
    adv_std_logs.append(float(np.std(advantages)))

    # update agent
    agent.update(states_buf, actions_buf, logps_buf, returns, advantages)

    rewards_per_episode.append(total_reward)
    actor_losses.append(agent.last_actor_loss)
    critic_losses.append(agent.last_critic_loss)
    entropy_logs.append(agent.last_entropy)

    if episode % 1500 == 0:
        print(f"Episode {episode}, Reward={total_reward:.2f}, ActorLoss={agent.last_actor_loss:.4f}, CriticLoss={agent.last_critic_loss:.4f}, Entropy={agent.last_entropy:.4f}")


In [ ]:

# --- Plots ---
plt.plot(rewards_per_episode)
plt.xlabel("Episode")
plt.ylabel("Total Reward")
plt.show()


In [ ]:

plt.plot(actor_losses, label="actor loss")
plt.xlabel("Episode")
plt.ylabel("Loss")
plt.legend()
plt.show()


In [ ]:

plt.plot(critic_losses, label="critic loss")
plt.xlabel("Episode")
plt.ylabel("Loss")
plt.legend()
plt.show()


In [ ]:
plt.figure()
plt.plot(entropy_logs, label="entropy")
plt.xlabel("Episode")
plt.ylabel("Entropy")
plt.legend()
plt.show()

In [ ]:
plt.figure()
plt.plot(adv_mean_logs, label="adv mean")
plt.plot(adv_std_logs, label="adv std")
plt.xlabel("Episode")
plt.legend()
plt.show()

In [ ]:
# Save / Load
def save_ppo(agent, filename="ppo_agent.pth"):
    torch.save({
        'actor_state_dict': agent.actor.state_dict(),
        'critic_state_dict': agent.critic.state_dict(),
        'optimizer_state_dict': agent.optimizer.state_dict()
    }, filename)
    print(f"PPO agent saved to {filename}")

def load_ppo(agent, filename="ppo_agent.pth"):
    checkpoint = torch.load(filename)
    agent.actor.load_state_dict(checkpoint['actor_state_dict'])
    agent.critic.load_state_dict(checkpoint['critic_state_dict'])
    agent.optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    print(f"PPO agent loaded from {filename}")


In [ ]:

save_ppo(agent)


In [ ]:
stats_df = pd.DataFrame({
    'PPO rewards': rewards_per_episode,
    'PPO actor loss': actor_losses,
    'PPO critic loss': critic_losses,
    'PPO entropy': entropy_logs
})
stats_df.to_excel("ppo_stats.xlsx", index = False)
print("Data exported to ppo_stats.xlsx")